# Recipe Clustering V9 – PCA vs t-SNE Visualization Test

**Purpose:** Test whether switching from t-SNE to PCA for the scatter plot resolves the outlier problem seen in V7/Model-2.

### Hypothesis
- V7 uses t-SNE with `perplexity=5`, which is far too small and scatters unique recipes (185.028, 188.740) arbitrarily far from the main clusters.
- PCA produces a linear, globally-stable F1/F2 biplot (like the hand-drawn reference image) where every recipe gets a meaningful position relative to the cluster structure.

### What we test
1. **Side-by-side**: t-SNE (V7 behavior) vs PCA (new) for Model-2 (as is × Grandfamilien)
2. **PCA biplot** with F1/F2 variance-explained axis labels
3. **Outlier diagnosis**: where do 185.028 and 188.740 land in PCA space and which cluster centroid are they closest to?
4. **Loading vectors**: which odour-type dimensions drive F1 and F2
5. **All 4 models** on PCA to pick the best one

## Setup

In [ ]:
import sys, os
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, PROJECT_ROOT)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

from sklearn.metrics import silhouette_score
from sklearn.preprocessing import normalize
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA

try:
    import faiss
    FAISS_AVAILABLE = True
    print('FAISS available')
except ImportError:
    FAISS_AVAILABLE = False
    print('FAISS not available – cannot run this notebook')

%matplotlib inline
plt.rcParams['figure.dpi'] = 120
print('Libraries loaded ✓')

## 1. Load & Preprocess (identical to V7)

In [ ]:
DATA_PATH   = '../data/gold/Versuchsdaten_3_1.csv'
IGNORE_PATH = '../data/gold/ignone_substances.csv'
CAS_PATH    = '../data/gold/CAS Nummern.csv'
OUTPUT_DIR  = '../outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

df_raw = pd.read_csv(DATA_PATH)
ign    = pd.read_csv(IGNORE_PATH)
cas    = pd.read_csv(CAS_PATH, header=13)

print(f'Versuchsdaten_3_1: {df_raw.shape[0]} rows, {df_raw["Rez.-Nr."].nunique()} recipes')
print(f'Ignore substances: {len(ign)} entries')
print(f'CAS lookup table : {len(cas)} rows')

In [ ]:
# Resolve ignore list → CAS numbers
ign_cas = ign[['Ident']].merge(
    cas[['Ident.', 'CAS-Nr.: - Hinweis 1']].rename(columns={'Ident.': 'Ident'}),
    on='Ident', how='left'
)
cas_to_ignore   = set(ign_cas['CAS-Nr.: - Hinweis 1'].dropna().astype(str).str.strip())
names_to_ignore = {str(n).lower().strip() for n in ign['Name']}

# Zero out ignored substances
df = df_raw.copy()
df['_cas'] = df['CAS-Nr.: - Hinweis 1'].astype(str).str.strip()
mask_cas = df['_cas'].isin(cas_to_ignore)
df.loc[mask_cas, 'Totalmenge'] = 0.0
df.drop(columns='_cas', inplace=True)
mask_name = df['Name'].str.lower().str.strip().isin(names_to_ignore)
df.loc[mask_name, 'Totalmenge'] = 0.0

# Re-normalise per recipe to sum = 1
per_recipe_total = df.groupby('Rez.-Nr.')['Totalmenge'].transform('sum')
df['Totalmenge'] = np.where(per_recipe_total > 0,
                            df['Totalmenge'] / per_recipe_total,
                            df['Totalmenge'])
sums = df.groupby('Rez.-Nr.')['Totalmenge'].sum()
assert sums.round(6).eq(1.0).all(), 'Not all recipes sum to 1!'
print('Preprocessing done ✓')
print(f'Active ingredient rows: {(df["Totalmenge"] > 0).sum()} / {len(df)}')

## 2. Feature Helpers (identical to V7)

In [ ]:
OT1 = 'Odour Type 1 FlavourWheel'
OT2 = 'Odour Type 2 Flavour Wheel'
OT3 = 'Odour Type 3 Flavour Wheel'
THRESHOLD_COL = 'Threshold ppm (Datenbank)'

def pos_weight(position, n_cols):
    return (n_cols + 1 - position) / n_cols

def thresh_factor(threshold_ppm, fallback=1.0):
    try:
        raw = str(threshold_ppm).strip().replace(',', '.')
        t = float(raw)
        return 1.0 / t if (not np.isnan(t) and t > 0) else fallback
    except (TypeError, ValueError):
        return fallback

def norm_term(term):
    if pd.isna(term) or not isinstance(term, str):
        return None
    t = term.lower().strip().replace('"', '').replace("'", '').rstrip('.,;:')
    return t if len(t) >= 2 else None

def build_vocabulary(df, feature_cols):
    all_terms = set()
    for col in feature_cols:
        if col in df.columns:
            for t in df[col].dropna().map(norm_term):
                if t:
                    all_terms.add(t)
    return sorted(all_terms)

def build_recipe_vectors(df, recipes, feature_cols_weighted, use_threshold):
    feature_cols = [col for col, _ in feature_cols_weighted]
    vocab = build_vocabulary(df, feature_cols)
    vocab_to_idx = {t: i for i, t in enumerate(vocab)}
    n_recipes = len(recipes)
    n_feat    = len(vocab)
    vectors   = np.zeros((n_recipes, n_feat), dtype=np.float64)
    for r_idx, recipe in enumerate(recipes):
        rows = df[df['Rez.-Nr.'] == recipe]
        for _, row in rows.iterrows():
            qty = float(row['Totalmenge'])
            if qty <= 0:
                continue
            t_fac     = thresh_factor(row[THRESHOLD_COL]) if use_threshold else 1.0
            ingr_base = qty * t_fac
            for col, col_weight in feature_cols_weighted:
                if col not in df.columns:
                    continue
                term = norm_term(row.get(col))
                if term and term in vocab_to_idx:
                    vectors[r_idx, vocab_to_idx[term]] += col_weight * ingr_base
    return vocab, normalize(vectors)

def run_faiss_clustering(recipe_vecs, k_range=(3, 12)):
    vectors_f32 = np.ascontiguousarray(recipe_vecs.astype('float32'))
    n_v, d = vectors_f32.shape
    scores = []
    best_k, best_score = k_range[0], -1
    best_labels, best_centroids = None, None
    for k in range(k_range[0], min(k_range[1] + 1, n_v)):
        km = faiss.Kmeans(d, k, niter=50, verbose=False, seed=42)
        km.train(vectors_f32)
        _, lbl = km.index.search(vectors_f32, 1)
        lbl = lbl.flatten()
        if len(set(lbl)) > 1:
            s = silhouette_score(vectors_f32, lbl)
            scores.append((k, s))
            if s > best_score:
                best_score     = s
                best_k         = k
                best_labels    = lbl.copy()
                best_centroids = km.centroids.copy()
    return best_labels, best_k, best_score, best_centroids, scores

def generate_cluster_names(cluster_labels, recipe_vecs, vocab, top_n=3):
    unique_labels = sorted(set(cluster_labels))
    global_cen    = recipe_vecs.mean(axis=0)
    centroids = {l: recipe_vecs[cluster_labels == l].mean(axis=0)
                 for l in unique_labels if l != -1}
    names = {}
    for label in unique_labels:
        if label == -1:
            names[label] = 'Outliers'
            continue
        cen     = centroids[label]
        dist    = cen - global_cen * 0.8
        top_idx = np.argsort(dist)[-6:][::-1]
        terms   = [vocab[i].capitalize()
                   for i in top_idx if dist[i] > 0 and cen[i] > 0.05][:top_n]
        if len(terms) < 2:
            terms = [vocab[i].capitalize()
                     for i in np.argsort(cen)[-top_n:][::-1]]
        names[label] = '-'.join(terms[:top_n])
    return names

def get_details(cluster_labels, cluster_names, recipe_vecs, vocab, recipes):
    details = {}
    for label in sorted(set(cluster_labels)):
        mask    = cluster_labels == label
        vecs    = recipe_vecs[mask]
        cen     = vecs.mean(axis=0)
        top_idx = np.argsort(cen)[-10:][::-1]
        details[label] = {
            'name'     : cluster_names.get(label, f'C{label}'),
            'size'     : int(mask.sum()),
            'recipes'  : [r for r, m in zip(recipes, mask) if m],
            'top_terms': [(vocab[i], round(float(cen[i]), 4)) for i in top_idx],
        }
    return details

print('Helpers defined ✓')

## 3. Model-2: t-SNE vs PCA Side-by-Side

We reproduce Model-2 from V7 (`as is × Grandfamilien`) and compare the two visualizations.

In [ ]:
recipes = df['Rez.-Nr.'].unique().tolist()
print(f'Total recipes: {len(recipes)}')

# Build vectors – Model 2 configuration (as is × Grandfamilien)
vocab_m2, vecs_m2 = build_recipe_vectors(
    df, recipes,
    feature_cols_weighted=[(OT1, 1.0)],
    use_threshold=False,
)
print(f'Vocabulary: {vocab_m2}')
print(f'Vector shape: {vecs_m2.shape}')

labels_m2, k_m2, score_m2, centroids_m2, _ = run_faiss_clustering(vecs_m2)
cnames_m2  = generate_cluster_names(labels_m2, vecs_m2, vocab_m2)
details_m2 = get_details(labels_m2, cnames_m2, vecs_m2, vocab_m2, recipes)

print(f'\nBest k={k_m2}, silhouette={score_m2:.4f}')
for l in sorted(set(labels_m2)):
    print(f'  Cluster {l} ({cnames_m2[l]}): {details_m2[l]["recipes"]}')

In [ ]:
OUTLIER_RECIPES = ['185.028', '188.740']
colors_m2 = plt.cm.Set2(np.linspace(0, 1, max(len(set(labels_m2)), 8)))

# --- t-SNE (V7 original) ---
perp_tsne = min(5, len(recipes) - 1)
tsne_coords = TSNE(n_components=2, perplexity=perp_tsne,
                   random_state=42, max_iter=1000).fit_transform(vecs_m2)

# --- PCA (new) ---
pca2 = PCA(n_components=2)
pca_coords = pca2.fit_transform(vecs_m2)
var_pct = pca2.explained_variance_ratio_ * 100

print(f't-SNE perplexity: {perp_tsne}')
print(f'PCA variance explained: F1={var_pct[0]:.2f}%  F2={var_pct[1]:.2f}%  total={sum(var_pct):.2f}%')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

for ax_idx, (ax, coords, xlabel, ylabel, title) in enumerate([
    (axes[0], tsne_coords,
     't-SNE Dim 1', 't-SNE Dim 2',
     f't-SNE  (perplexity={perp_tsne})  ← V7 original'),
    (axes[1], pca_coords,
     f'F1 ({var_pct[0]:.2f} %)', f'F2 ({var_pct[1]:.2f} %)',
     f'PCA  (total variance {sum(var_pct):.2f} %)  ← NEW'),
]):
    for j, label in enumerate(sorted(set(labels_m2))):
        mask = labels_m2 == label
        pts  = coords[mask]
        c    = colors_m2[j % len(colors_m2)]
        ax.scatter(pts[:, 0], pts[:, 1], c=[c], s=160, alpha=0.85,
                   edgecolors='black', lw=0.6,
                   label=f'C{label}: {cnames_m2[label]}')

    for i, rec in enumerate(recipes):
        # Highlight the two outlier recipes
        is_outlier = rec in OUTLIER_RECIPES
        weight = 'bold' if is_outlier else 'normal'
        color  = 'red'  if is_outlier else 'black'
        ax.annotate(
            rec,
            (coords[i, 0], coords[i, 1]),
            fontsize=7 if is_outlier else 6,
            fontweight=weight, color=color,
            ha='center', va='bottom', alpha=0.9
        )
        if is_outlier:
            ax.scatter(coords[i, 0], coords[i, 1],
                       marker='*', s=280, c='red', zorder=5,
                       edgecolors='darkred', lw=0.8)

    ax.set_xlabel(xlabel, fontsize=10)
    ax.set_ylabel(ylabel, fontsize=10)
    ax.set_title(f'Model-2: as is × Grandfamilien\n{title}',
                 fontsize=11, fontweight='bold')
    ax.legend(loc='upper left', fontsize=8, framealpha=0.9)
    ax.axhline(0, color='gray', lw=0.5, ls='--')
    ax.axvline(0, color='gray', lw=0.5, ls='--')
    ax.grid(True, alpha=0.2)

plt.suptitle('t-SNE vs PCA – Model-2 (as is × Grandfamilien)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/v9_tsne_vs_pca_model2.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → v9_tsne_vs_pca_model2.png')

## 4. PCA Outlier Diagnosis

Where do 185.028 and 188.740 land in PCA space? Which cluster centroid are they closest to?

In [ ]:
print('=== PCA Outlier Diagnosis ===')
print(f'Vocabulary (Grandfamilien): {vocab_m2}\n')

# Raw (non-normalised) vectors per recipe to see ingredient composition
_, vecs_raw = build_recipe_vectors(
    df, recipes,
    feature_cols_weighted=[(OT1, 1.0)],
    use_threshold=False,
)

for rec in OUTLIER_RECIPES:
    if rec not in recipes:
        print(f'{rec}: NOT FOUND in dataset')
        continue
    idx = recipes.index(rec)
    label = labels_m2[idx]
    pca_pos = pca_coords[idx]

    print(f'Recipe {rec}:')
    print(f'  Assigned cluster : C{label} ({cnames_m2[label]})')
    print(f'  PCA position     : F1={pca_pos[0]:+.4f}  F2={pca_pos[1]:+.4f}')

    # Distances to all cluster centroids (in original vector space)
    vec = vecs_m2[idx]
    print('  Distance to each cluster centroid (cosine similarity):')
    for l in sorted(set(labels_m2)):
        cen = vecs_m2[labels_m2 == l].mean(axis=0)
        sim = float(np.dot(vec, cen) / (np.linalg.norm(vec) * np.linalg.norm(cen) + 1e-12))
        marker = ' ← assigned' if l == label else ''
        print(f'    C{l} ({cnames_m2[l]:30s}): {sim:.4f}{marker}')

    # Ingredient breakdown
    rows = df[(df['Rez.-Nr.'] == rec) & (df['Totalmenge'] > 0)].copy()
    rows = rows.sort_values('Totalmenge', ascending=False)
    print(f'  Top ingredients (by share):')
    for _, row in rows.head(8).iterrows():
        ot1 = row.get(OT1, 'n/a')
        ot2 = row.get(OT2, 'n/a')
        print(f'    {row["Totalmenge"]:.3f}  {row.get("Name",""):<45}  OT1={ot1}  OT2={ot2}')
    print()

## 5. PCA Loading Vectors

Which odour dimensions drive F1 and F2? This replicates the PCA biplot arrow interpretation.

In [ ]:
loadings = pca2.components_  # shape: (2, n_vocab)

print('PCA Loadings (contribution of each odour term to F1 and F2):')
print(f'{"Term":<20} {"F1 loading":>12} {"F2 loading":>12}')
print('-' * 46)
for i, term in enumerate(vocab_m2):
    print(f'{term:<20} {loadings[0, i]:>+12.4f} {loadings[1, i]:>+12.4f}')

# PCA biplot with loading arrows
fig, ax = plt.subplots(figsize=(11, 9))

for j, label in enumerate(sorted(set(labels_m2))):
    mask = labels_m2 == label
    pts  = pca_coords[mask]
    c    = colors_m2[j % len(colors_m2)]
    ax.scatter(pts[:, 0], pts[:, 1], c=[c], s=160, alpha=0.85,
               edgecolors='black', lw=0.6, zorder=3,
               label=f'C{label}: {cnames_m2[label]}')

for i, rec in enumerate(recipes):
    is_outlier = rec in OUTLIER_RECIPES
    ax.annotate(
        rec, (pca_coords[i, 0], pca_coords[i, 1]),
        fontsize=7 if is_outlier else 6,
        fontweight='bold' if is_outlier else 'normal',
        color='red' if is_outlier else 'black',
        ha='center', va='bottom', alpha=0.9
    )
    if is_outlier:
        ax.scatter(pca_coords[i, 0], pca_coords[i, 1],
                   marker='*', s=300, c='red', zorder=6,
                   edgecolors='darkred', lw=0.8)

# Loading arrows – scale to fit the plot
scale = 0.6 * max(np.abs(pca_coords).max(), 1e-6) / (np.abs(loadings).max() + 1e-9)
for i, term in enumerate(vocab_m2):
    dx, dy = loadings[0, i] * scale, loadings[1, i] * scale
    ax.annotate('', xy=(dx, dy), xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color='navy', lw=1.8))
    ax.text(dx * 1.12, dy * 1.12, term.capitalize(),
            fontsize=9, color='navy', fontweight='bold',
            ha='center', va='center')

ax.axhline(0, color='gray', lw=0.5, ls='--')
ax.axvline(0, color='gray', lw=0.5, ls='--')
ax.set_xlabel(f'F1 ({var_pct[0]:.2f} %)', fontsize=11)
ax.set_ylabel(f'F2 ({var_pct[1]:.2f} %)', fontsize=11)
ax.set_title('PCA Biplot – Model-2: as is × Grandfamilien\n'
             'Arrows = odour loading directions  |  ★ = Focus recipes (185.028, 188.740)',
             fontsize=11, fontweight='bold')
ax.legend(loc='upper left', fontsize=8, framealpha=0.9)
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/v9_pca_biplot_model2.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → v9_pca_biplot_model2.png')

## 6. All 4 Models – PCA Comparison

Same 4 model configs as V7, but every scatter plot now uses PCA instead of t-SNE.

In [ ]:
MODEL_CONFIGS = [
    {
        'name': 'as is × gewichtete Subfamilien',
        'feature_cols_weighted': [
            (OT1, pos_weight(1, 4)),
            (OT2, pos_weight(2, 4)),
            (OT3, pos_weight(3, 4)),
        ],
        'use_threshold': False,
    },
    {
        'name': 'as is × Grandfamilien',
        'feature_cols_weighted': [(OT1, 1.0)],
        'use_threshold': False,
    },
    {
        'name': 'as is/Threshold × gewichtete Subfamilien',
        'feature_cols_weighted': [
            (OT1, pos_weight(1, 4)),
            (OT2, pos_weight(2, 4)),
            (OT3, pos_weight(3, 4)),
        ],
        'use_threshold': True,
    },
    {
        'name': 'as is/Threshold × Grandfamilien',
        'feature_cols_weighted': [(OT1, 1.0)],
        'use_threshold': True,
    },
]

results_v9 = {}

for i_cfg, cfg in enumerate(MODEL_CONFIGS, 1):
    name = cfg['name']
    print(f'\n{"-"*60}')
    print(f'Model {i_cfg}: {name}')

    vocab, vecs = build_recipe_vectors(
        df, recipes,
        feature_cols_weighted=cfg['feature_cols_weighted'],
        use_threshold=cfg['use_threshold'],
    )

    labels, best_k, best_score, _, scores = run_faiss_clustering(vecs)
    cnames  = generate_cluster_names(labels, vecs, vocab)
    details = get_details(labels, cnames, vecs, vocab, recipes)

    pca_obj  = PCA(n_components=2)
    coords   = pca_obj.fit_transform(vecs)
    var_expl = pca_obj.explained_variance_ratio_ * 100

    results_v9[name] = {
        'labels' : labels, 'k': best_k, 'score': best_score,
        'vocab'  : vocab,  'vecs': vecs,
        'cnames' : cnames, 'details': details,
        'pca_coords': coords, 'var_expl': var_expl,
        'pca_obj': pca_obj,
    }

    print(f'  k={best_k}, silhouette={best_score:.4f}, '
          f'PCA F1={var_expl[0]:.1f}% F2={var_expl[1]:.1f}%')
    for l in sorted(set(labels)):
        recs = details[l]['recipes']
        flag = ' ⚠ outlier recipes here!' if any(r in OUTLIER_RECIPES for r in recs) else ''
        print(f'    C{l} ({cnames[l]}): {recs}{flag}')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(20, 16))
axes = axes.flatten()

for i, (name, res) in enumerate(results_v9.items()):
    ax      = axes[i]
    coords  = res['pca_coords']
    labels  = res['labels']
    cnames  = res['cnames']
    ve      = res['var_expl']
    unique_labels = sorted(set(labels))
    colors = plt.cm.Set2(np.linspace(0, 1, max(len(unique_labels), 8)))

    for j, label in enumerate(unique_labels):
        mask = labels == label
        pts  = coords[mask]
        c    = colors[j % len(colors)]
        ax.scatter(pts[:, 0], pts[:, 1], c=[c], s=130, alpha=0.85,
                   edgecolors='black', lw=0.5,
                   label=f'C{label}: {cnames.get(label, "")[:22]} ({mask.sum()}r)')

    for k_r, rec in enumerate(recipes):
        is_out = rec in OUTLIER_RECIPES
        ax.annotate(
            rec, (coords[k_r, 0], coords[k_r, 1]),
            fontsize=7.5 if is_out else 5.5,
            fontweight='bold' if is_out else 'normal',
            color='red' if is_out else 'dimgray',
            ha='center', va='bottom', alpha=0.95
        )
        if is_out:
            ax.scatter(coords[k_r, 0], coords[k_r, 1],
                       marker='*', s=260, c='red', zorder=5,
                       edgecolors='darkred', lw=0.8)

    ax.axhline(0, color='gray', lw=0.4, ls='--')
    ax.axvline(0, color='gray', lw=0.4, ls='--')
    ax.set_xlabel(f'F1 ({ve[0]:.2f} %)', fontsize=9)
    ax.set_ylabel(f'F2 ({ve[1]:.2f} %)', fontsize=9)
    ax.set_title(f'Model {i+1}: {name}\nPCA – total variance {sum(ve):.1f}%',
                 fontsize=9, fontweight='bold')
    ax.legend(loc='upper left', fontsize=6.5, framealpha=0.9)
    ax.grid(True, alpha=0.2)

plt.suptitle('All 4 Models – PCA Scatter (★ = 185.028 & 188.740)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/v9_pca_all_models.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → v9_pca_all_models.png')

## 7. Verdict: Did PCA Fix the Outlier Problem?

Print a clear summary comparing cluster assignments and positions.

In [ ]:
print('=' * 70)
print('VERDICT – Outlier recipe positions across all 4 models (PCA)')
print('=' * 70)

for i, (name, res) in enumerate(results_v9.items(), 1):
    print(f'\nModel {i}: {name}')
    print(f'  k={res["k"]}, silhouette={res["score"]:.4f}')
    for rec in OUTLIER_RECIPES:
        if rec not in recipes:
            print(f'  {rec}: NOT IN DATASET')
            continue
        idx   = recipes.index(rec)
        label = res['labels'][idx]
        cname = res['cnames'][label]
        pos   = res['pca_coords'][idx]
        # Cosine similarity to its assigned centroid
        vec   = res['vecs'][idx]
        cen   = res['vecs'][res['labels'] == label].mean(axis=0)
        sim   = float(np.dot(vec, cen) / (np.linalg.norm(vec) * np.linalg.norm(cen) + 1e-12))
        print(f'  {rec} → C{label} ({cname:30s})  '
              f'F1={pos[0]:+.3f} F2={pos[1]:+.3f}  cos_sim={sim:.3f}')

print()
print('Comparison with V7 (t-SNE, perplexity=5):')
print('  t-SNE scattered 185.028 and 188.740 to arbitrary positions.')
print('  PCA places them at stable, interpretable coordinates relative to cluster structure.')
print()
print('Note: Cluster ASSIGNMENT is unchanged (FAISS k-means is the same).')
print('      PCA only changes what you SEE – not which cluster each recipe belongs to.')
print('      If 185.028 or 188.740 still appear isolated, they are genuinely atypical')
print('      recipes with unusual ingredient profiles for strawberry aroma.')

## 8. Summary

| Aspect | V7 (t-SNE) | V9 (PCA) |
|--------|-----------|----------|
| **Axes** | Arbitrary Dim 1/2 | F1/F2 with % variance explained |
| **Perplexity** | 5 (too small) | N/A |
| **Stability** | Stochastic, changes each run | Deterministic |
| **Global structure** | Distorted | Preserved |
| **Outlier scatter** | Exaggerated by low perplexity | Reflects true distance in feature space |
| **Cluster assignment** | FAISS k-means (unchanged) | FAISS k-means (unchanged) |
| **Loading interpretation** | Not available | Arrow directions show which odour type drives each axis |